# K1 · Hybrid RAG đầy đủ: BM25 + dense bge-m3 + RRF + reranker (T09)
Hoàn thiện ablation R1–R5 mà máy CPU đã đánh dấu `skipped`. ~10–15 phút trên T4.

In [ ]:
%%capture
!pip install -q "sentence-transformers>=3.0" rank-bm25

In [ ]:
import sys, glob, json, os, time
sys.path.insert(0, glob.glob('/kaggle/input/**/code/src', recursive=True)[0])
from kimochi.kaggle_env import setup, gpu_info
ROOT = setup()
import pandas as pd
from pathlib import Path
RES = ROOT / 'results'
print(gpu_info())

In [ ]:
from kimochi import retriever
for split in ['val', 'test_synth']:
    res = retriever.run(split)
    display(res.round(3))

Nếu đã có `test_gold.jsonl` có trường `gold_doc_ids`, đánh giá thêm trên bộ viết tay (số chính thức cho báo cáo):

In [ ]:
gold = ROOT / 'data/splits/test_gold.jsonl'
if gold.exists():
    import numpy as np
    g = pd.read_json(gold, lines=True)
    g = g[g.gold_doc_ids.map(lambda x: bool(x))]
    dense, rer = retriever.load_models()
    r = retriever.HybridRetriever(retriever.load_docs(), dense, rer)
    rows = []
    for cfg in retriever.CONFIGS:
        try:
            ranked = [r.search(q, cfg, k=10) for q in g['query']]
        except Exception as e:
            continue
        # nhiều tài liệu đúng: lấy vị trí tài liệu đúng đầu tiên
        first = [next((d for d in rk if d in set(gd)), gd[0]) for rk, gd in zip(ranked, g.gold_doc_ids)]
        rows.append({'config': cfg, **retriever.metrics(ranked, first)})
    out = pd.DataFrame(rows); out.to_csv(RES / 'retrieval_test_gold.csv', index=False); display(out.round(3))
else:
    print('Chưa có test_gold.jsonl — bỏ qua')